# 🎙️ Urdu Voice — Urdu voiceover in your own voice

1. **Runtime → Change runtime type → T4 GPU → Save**
2. Run the 4 cells in order (▶). The last cell prints a **public link** (`https://….gradio.live`) — open it on phone or PC.
3. Voices and outputs are saved in Google Drive → `MyDrive/UrduVoice/`.

The link works only while this Colab tab is running (free tier stops after idle time — then just run the cells again).

In [ ]:
# 1) Connect Google Drive (saves your voice + outputs)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2) Install (2-4 min)
!pip -q install omnivoice soundfile gradio
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch runtime to T4 GPU!')

In [ ]:
%%writefile /content/app.py
# Urdu Voice — Gradio app (runs inside Colab, cell 3 writes and runs this file)
import os, re, glob, time, subprocess, numpy as np, soundfile as sf, torch, gradio as gr
from omnivoice import OmniVoice

SR = 24000
BASE = "/content/drive/MyDrive/UrduVoice" if os.path.isdir("/content/drive/MyDrive") else "/content/UrduVoice"
VOICES, OUTS = f"{BASE}/voices", f"{BASE}/outputs"
os.makedirs(VOICES, exist_ok=True); os.makedirs(OUTS, exist_ok=True)
REF_TEXT = "السلام علیکم! آج ہم اردو میں اپنا پہلا وائس اوور بنا رہے ہیں۔"

model = OmniVoice.from_pretrained("k2-fsa/OmniVoice", device_map="cuda:0", dtype=torch.float16)
STATE = {"chunks": [], "audio": [], "voice": None}

def voice_list():
    return sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f"{VOICES}/*.wav"))

def save_voice(path, name):
    name = re.sub(r"[^\w؀-ۿ-]+", "_", (name or "").strip())
    if not path: return "پہلے ریکارڈ کریں / Record first", gr.update()
    if not name: return "نام لکھیں / Enter a name", gr.update()
    out = f"{VOICES}/{name}.wav"
    # mono 24 kHz, max 15 s, trim leading/trailing silence
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", path, "-ac", "1", "-ar", str(SR),
                    "-af", "silenceremove=start_periods=1:start_threshold=-45dB,areverse,"
                           "silenceremove=start_periods=1:start_threshold=-45dB,areverse",
                    "-t", "15", out], check=True)
    d = sf.info(out).duration
    msg = f"✓ محفوظ: {name} ({d:.1f}s)" + ("  ⚠ بہت چھوٹا، 5-10 سیکنڈ بہتر" if d < 3 else "")
    return msg, gr.update(choices=voice_list(), value=name)

def split(text, limit=250):
    parts = [p.strip(" .\t") for p in re.split(r"(?<=[۔.!؟?…])\s+|\n+", text) if p.strip(" .\t")]
    chunks, cur = [], ""
    for p in parts:
        if cur and len(cur) + len(p) > limit:
            chunks.append(cur); cur = p
        else:
            cur = (cur + " " + p).strip()
    if cur: chunks.append(cur)
    return chunks

def gen_one(text):
    a = model.generate(text=text, ref_audio=STATE["voice"], ref_text=REF_TEXT)[0]
    return a.astype(np.float32)

def export(speed, pause_s):
    pause = np.zeros(int(SR * pause_s), dtype=np.float32)
    full = np.concatenate([x for a in STATE["audio"] for x in (a, pause)])
    stamp = time.strftime("%Y%m%d_%H%M%S")
    raw, wav, mp3 = f"/tmp/raw_{stamp}.wav", f"{OUTS}/voiceover_{stamp}.wav", f"{OUTS}/voiceover_{stamp}.mp3"
    sf.write(raw, full, SR)
    af = ["-af", f"atempo={speed}"] if abs(speed - 1) > 0.01 else []
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", raw, *af, wav], check=True)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", wav, "-b:a", "128k", mp3], check=True)
    mins = sf.info(wav).duration / 60
    table = "\n".join(f"{i}. {c[:60]}…" for i, c in enumerate(STATE["chunks"], 1))
    return wav, mp3, f"✓ {len(STATE['chunks'])} حصے، {mins:.1f} منٹ — Drive: {OUTS}", table

def generate(voice, text, speed, pause_s, progress=gr.Progress()):
    if not voice: raise gr.Error("پہلے آواز منتخب کریں / Select a voice")
    if not text.strip(): raise gr.Error("متن لکھیں / Enter text")
    STATE["voice"] = f"{VOICES}/{voice}.wav"
    STATE["chunks"] = split(text)
    STATE["audio"] = []
    for i, c in enumerate(STATE["chunks"]):
        progress(i / len(STATE["chunks"]), desc=f"حصہ {i+1}/{len(STATE['chunks'])}")
        STATE["audio"].append(gen_one(c))
    return export(speed, pause_s)

def redo(num, new_text, speed, pause_s):
    n = int(num or 0)
    if not STATE["audio"] or not 1 <= n <= len(STATE["chunks"]):
        raise gr.Error("غلط حصہ نمبر / Invalid part number")
    if new_text and new_text.strip():
        STATE["chunks"][n - 1] = new_text.strip()
    STATE["audio"][n - 1] = gen_one(STATE["chunks"][n - 1])
    return export(speed, pause_s)

CSS = ".urdu textarea, .urdu input {direction: rtl; font-family: 'Noto Nastaliq Urdu', serif; font-size: 20px; line-height: 2.2}"
HEAD = '<link href="https://fonts.googleapis.com/css2?family=Noto+Nastaliq+Urdu&display=swap" rel="stylesheet">'

with gr.Blocks(title="Urdu Voice", css=CSS, head=HEAD) as app:
    gr.Markdown("# 🎙️ Urdu Voice — اپنی آواز میں اردو وائس اوور\nصرف **اپنی** آواز کلون کریں۔ Only clone your own voice.")
    with gr.Tab("1) آواز ریکارڈ کریں / Record voice"):
        gr.Markdown(f"## یہ جملہ صاف آواز میں پڑھیں (5–10 سیکنڈ، خاموش کمرہ):\n\n<div dir='rtl' style='font-size:26px;font-family:Noto Nastaliq Urdu;line-height:2.4'>{REF_TEXT}</div>")
        mic = gr.Audio(sources=["microphone", "upload"], type="filepath", label="ریکارڈ / Record or upload")
        vname = gr.Textbox(label="آواز کا نام / Voice name", placeholder="e.g. yasir")
        save_btn = gr.Button("محفوظ کریں / Save voice", variant="primary")
        save_msg = gr.Markdown()
    with gr.Tab("2) وائس اوور بنائیں / Make voiceover"):
        with gr.Row():
            voice = gr.Dropdown(voice_list(), label="آواز / Voice", value=(voice_list() or [None])[0])
            refresh = gr.Button("🔄", scale=0)
        text = gr.Textbox(lines=12, label="اردو متن / Urdu script", elem_classes="urdu")
        with gr.Row():
            speed = gr.Slider(0.8, 1.25, 1.0, step=0.05, label="رفتار / Speed")
            pause_s = gr.Slider(0.1, 1.0, 0.35, step=0.05, label="وقفہ (سیکنڈ) / Pause between parts")
        go = gr.Button("▶ بنائیں / Generate", variant="primary")
        status = gr.Markdown()
        player = gr.Audio(label="سنیں / Listen (WAV)", type="filepath")
        mp3 = gr.File(label="MP3 ڈاؤن لوڈ (WhatsApp کے لیے) / Download MP3")
        with gr.Accordion("کوئی حصہ دوبارہ بنائیں / Redo one part", open=False):
            parts = gr.Textbox(label="حصے / Parts", lines=8, interactive=False, elem_classes="urdu")
            num = gr.Number(label="حصہ نمبر / Part number", precision=0)
            new_text = gr.Textbox(label="نیا متن (اختیاری) / New text for this part (optional)", elem_classes="urdu")
            redo_btn = gr.Button("دوبارہ بنائیں / Redo part")

    save_btn.click(save_voice, [mic, vname], [save_msg, voice])
    refresh.click(lambda: gr.update(choices=voice_list()), None, voice)
    go.click(generate, [voice, text, speed, pause_s], [player, mp3, status, parts])
    redo_btn.click(redo, [num, new_text, speed, pause_s], [player, mp3, status, parts])

app.queue().launch(share=True, debug=True)


In [ ]:
!python /content/app.py